In [5]:
import openai

from qdrant_client import QdrantClient

In [6]:
COLLECTION_NAME = "Recipes-collection-01"

In [7]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [8]:
def get_embedding(text, model="text-embedding-3-small"):
  response = openai.embeddings.create(
    input=text,
    model=model
  )
  return response.data[0].embedding

## Retrieval

Embed the query and ask Qdrant for the top-`k` most similar recipes.

We pull only a curated set of fields from each payload, not the full `text`: retrieval matches on the whole document, but the answer just needs the essentials. Keeping `id` / `score` / `rating` around also gives us what we need for citation, debugging and evals.

In [9]:
def retrieve_data(query, k=5):
  query_embedding = get_embedding(query)
  results = qdrant_client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_embedding,
    limit=k,
  )
  retrieved = []
  for result in results.points:
    payload = result.payload
    retrieved.append({
      "id": int(payload["RecipeId"]),
      "name": payload["Name"],
      "score": result.score,
      "rating": payload["bayesian_rating"],
      "n_ratings": payload["n_ratings"],
      "calories": payload.get("Calories"),
      "protein": payload.get("ProteinContent"),
      "carbs": payload.get("CarbohydrateContent"),
      "fat": payload.get("FatContent"),
      "total_time": payload.get("total_time_minutes"),
      "ingredients": payload.get("RecipeIngredientParts") or [],
      "instructions": payload.get("RecipeInstructions") or [],
    })
  return retrieved

In [10]:
retrieve_data("a creamy pasta with mushrooms", k=3)

[{'id': 245367,
  'name': 'Creamy Shiitake Linguine With Scallions/ Green Onions',
  'score': 0.6173099,
  'rating': 4.692244584372267,
  'n_ratings': 1,
  'calories': 808.2,
  'protein': 19.6,
  'carbs': 98.4,
  'fat': 35.9,
  'total_time': 25,
  'ingredients': ['linguine',
   'shiitake mushroom',
   'butter',
   'garlic cloves',
   'shallots',
   'dry white wine',
   'chicken broth',
   'heavy whipping cream',
   'nutmeg',
   'scallion',
   'lemon juice',
   'fresh thyme leaves'],
  'instructions': ['Cook linguine pasta according to package directions to al dente; drain, reserving some of the pasta water and return to pan to keep warm.',
   'In a large frying pan, saute mushrooms in butter until they begin to soften.',
   'Add minced garlic and shallots; sauté an additional 2 minutes.',
   'Add white wine and simmer until wine is reduced by half.',
   'Add 1/4 cup of chicken broth, heavy cream, and nutmeg; simmer until sauce thickens.',
   'Remove from heat; add scallions, lemon juic

## Context formatting

Turn the retrieved recipes into a single readable string for the prompt. This is where we decide **what the LLM actually sees**: a curated subset (name, rating, nutrition, time, ingredients, a short step preview), not the full procedure. Instructions are truncated to keep the prompt small — the full recipe stays one direct lookup away.

In [11]:
def process_context(retrieved, max_steps_chars=300):
  blocks = []
  for i, r in enumerate(retrieved, start=1):
    nutrition = []
    if r["calories"] is not None:
      nutrition.append(f"{round(r['calories'])} kcal")
    if r["protein"] is not None:
      nutrition.append(f"P {round(r['protein'])}g")
    if r["carbs"] is not None:
      nutrition.append(f"C {round(r['carbs'])}g")
    if r["fat"] is not None:
      nutrition.append(f"F {round(r['fat'])}g")

    ingredients = ", ".join(str(x) for x in r["ingredients"] if x)

    steps = " ".join(str(s) for s in r["instructions"] if s)
    if len(steps) > max_steps_chars:
      steps = steps[:max_steps_chars].rstrip() + "..."

    tt = r["total_time"]

    blocks.append(
      f"[{i}] {r['name']} (id: {r['id']})\n"
      f"  rating: {r['rating']:.1f} ({r['n_ratings']} reviews)\n"
      f"  nutrition: {' | '.join(nutrition) or 'n/a'}\n"
      f"  total time: {f'{tt} min' if tt is not None else 'n/a'}\n"
      f"  ingredients: {ingredients}\n"
      f"  steps: {steps}"
    )

  return "\n\n".join(blocks)

In [12]:
print(process_context(retrieve_data("a creamy pasta with mushrooms", k=3)))

[1] Creamy Shiitake Linguine With Scallions/ Green Onions (id: 245367)
  rating: 4.7 (1 reviews)
  nutrition: 808 kcal | P 20g | C 98g | F 36g
  total time: 25 min
  ingredients: linguine, shiitake mushroom, butter, garlic cloves, shallots, dry white wine, chicken broth, heavy whipping cream, nutmeg, scallion, lemon juice, fresh thyme leaves
  steps: Cook linguine pasta according to package directions to al dente; drain, reserving some of the pasta water and return to pan to keep warm. In a large frying pan, saute mushrooms in butter until they begin to soften. Add minced garlic and shallots; sauté an additional 2 minutes. Add white wine and sim...

[2] Penne With Mushroom Sauce (id: 128694)
  rating: 4.7 (0 reviews)
  nutrition: 541 kcal | P 19g | C 88g | F 8g
  total time: 106 min
  ingredients: butter, onion, celery, carrot, tomato paste, dry red wine, dried thyme, black pepper, water, cornstarch, salt
  steps: Melt 1 tablespoon butter in a nonstick skillet over medium-high heat. Ad

## Prompt

Build the **system prompt**: the role, the rules and the formatted recipes the model is allowed to use (the question is sent separately as the `user` message). The instructions are the **guard-rail**: they keep the model anchored to the recipes we actually retrieved (no inventing dishes) and teach it how to behave in our domain — respect constraints (calories, time, ingredients), be honest when nothing fits, and never present the truncated step preview as the full method.

In [21]:
def build_system_prompt(context):
  return f"""
You are a helpful cooking assistant. You help people decide what to cook by recommending recipes from the ones available below.

Instructions:
- Only recommend recipes from the available recipes. Never invent recipes, ingredients, or nutrition values.
- Refer to recipes by their name; you may add the id in parentheses so it can be looked up.
- If the question has constraints (calories, time, an ingredient to include or avoid, a meal type), respect them and prefer recipes that match.
- If none of the available recipes fit the request well, say so honestly instead of forcing a poor match.
- The steps shown are only a short preview, not the full method, so don't present them as complete instructions.
- Keep the answer concise and friendly. Do not use markdown.

Available recipes:
{context}
"""

In [22]:
question = "a creamy pasta with mushrooms"
print(build_system_prompt(process_context(retrieve_data(question, k=3))))


You are a helpful cooking assistant. You help people decide what to cook by recommending recipes from the ones available below.

Instructions:
- Only recommend recipes from the available recipes. Never invent recipes, ingredients, or nutrition values.
- Refer to recipes by their name; you may add the id in parentheses so it can be looked up.
- If the question has constraints (calories, time, an ingredient to include or avoid, a meal type), respect them and prefer recipes that match.
- If none of the available recipes fit the request well, say so honestly instead of forcing a poor match.
- The steps shown are only a short preview, not the full method, so don't present them as complete instructions.
- Keep the answer concise and friendly. Do not use markdown.

Available recipes:
[1] Creamy Shiitake Linguine With Scallions/ Green Onions (id: 245367)
  rating: 4.7 (1 reviews)
  nutrition: 808 kcal | P 20g | C 98g | F 36g
  total time: 25 min
  ingredients: linguine, shiitake mushroom, but

## Generation & pipeline

Send the prompt to the model — `system` holds the role, rules and retrieved recipes, `user` holds just the question — and read back the answer. `rag_pipeline` is only the glue: retrieve → format → build prompt → generate.

In [23]:
def generate_answer(system_prompt, question):
  response = openai.chat.completions.create(
    model="gpt-5.4-nano",
    messages=[
      {"role": "system", "content": system_prompt},
      {"role": "user", "content": question},
    ],
    reasoning_effort="none",
  )
  return response.choices[0].message.content


def rag_pipeline(question, k=5):
  context = process_context(retrieve_data(question, k))
  system_prompt = build_system_prompt(context)
  return generate_answer(system_prompt, question)

In [24]:
print(rag_pipeline("a creamy pasta with mushrooms"))

Try Creamy Shiitake Linguine With Scallions/ Green Onions (245367). It’s a creamy, mushroom-forward pasta (shiitake in a white wine + chicken broth + heavy cream sauce) and takes about 25 minutes.


In [25]:
# Numeric-constraint query: pure semantic search can't enforce "< 100 calories".
print(rag_pipeline("a breakfast with less than 100 calories"))

You’ve got a couple good options under 100 calories:

1) Lower Carb Healthy Breakfast Parfait (246597) — 327 kcal (not under 100)
2) Extreme Low-Fat Buttermilk-Bran Breakfast Squares (167777) — 160 kcal (not under 100)
3) Yummy Low-Fat French Toast (241565) — 402 kcal (not under 100)
4) Oversized Carnival Oatmeal (525575) — 232 kcal (not under 100)
5) Low Fat Egg McMuffin (505904) — 141 kcal (not under 100)

None of the available recipes are actually under 100 calories based on the listed nutrition info. If you want, tell me your preferred meal style (sweet vs savory) and how strict “less than 100” is (e.g., under 150), and I’ll pick the closest match.
